In [ ]:
import sys
from pathlib import Path

def _find_project_root_by_src(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'src').is_dir():
            return parent
    return start

PROJECT_ROOT = _find_project_root_by_src(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import và setup
from src.data.loader import DocumentLoader
from src.data.chunker import TextChunker, ChunkStrategy
from src.pipeline.experiment_tracker import ExperimentTracker

tracker = ExperimentTracker()

ImportError: cannot import name 'TextChunker' from 'src.data.chunker' (/Users/hanhnguyen/AI-Research-Assistant-with-RAG/src/data/chunker.py)

In [ ]:
# Cell 2: Tải tài liệu mẫu
loader = DocumentLoader()
doc = loader.load("data/raw/sample_paper.pdf")
print(f"Nội dung: {len(doc.content)} ký tự")

In [ ]:
# Cell 3: Thực nghiệm Fixed-Size Chunking
chunker_fixed = TextChunker(strategy=ChunkStrategy.FIXED_SIZE, chunk_size=256)
chunks_fixed = chunker_fixed.chunk(doc)
print(f"Fixed-size: {len(chunks_fixed)} chunks")

In [ ]:
# Cell 4: Thực nghiệm Recursive Chunking
chunker_recursive = TextChunker(strategy=ChunkStrategy.RECURSIVE, chunk_size=512)
chunks_recursive = chunker_recursive.chunk(doc)
print(f"Recursive: {len(chunks_recursive)} chunks")

In [ ]:
# Cell 5: So sánh kết quả
def compare_strategies(chunks_a, name_a, chunks_b, name_b):
    """So sánh hai chiến lược chunking"""
    stats = {
        name_a: {
            "count": len(chunks_a),
            "avg_len": sum(len(c.content) for c in chunks_a) / len(chunks_a),
            "min_len": min(len(c.content) for c in chunks_a),
            "max_len": max(len(c.content) for c in chunks_a),
        },
        name_b: {
            "count": len(chunks_b),
            "avg_len": sum(len(c.content) for c in chunks_b) / len(chunks_b),
        }
    }
    return stats

stats = compare_strategies(chunks_fixed, "fixed_size", chunks_recursive, "recursive")

In [ ]:
# Cell 6: Log kết quả thực nghiệm
tracker.log_indexing(
    doc_id=doc.doc_id,
    chunk_strategy="fixed_size",
    chunk_size=256,
    num_chunks=len(chunks_fixed),
    latency_ms=0.0,
)